<a href="https://colab.research.google.com/github/selidrissi1/Superstore-discount-analysis/blob/main/%5B2026_2027%5D%5BDecision_Models%5D%5BGroup_1%5D_covering_problems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Set covering model

In [ ]:
# create a variable 'inf' to store a big number.
inf  = 9999

In [ ]:
# create the original distance matrix
distance = [[0, 8, 15, 10, inf, inf],
            [8, 0, 12, 7, 16, inf],
            [15, 12, 0, inf, 9, 11],
            [10, 7, inf, 0, 11, 17],
            [inf, 16, 9, 11, 0, 13],
            [inf, inf, 11, 17, 13, 0]]

In [ ]:
# Floyd-Warshall Algorithm: to find the shortest path from one point to another
# source: https://www.programiz.com/dsa/floyd-warshall-algorithm
def floyd_warshall(G):
    distance = list(map(lambda i: list(map(lambda j: j, i)), G))

    # get the number of nodes / list size
    nV = len(G)

    # Adding vertices individually
    for k in range(nV):
        for i in range(nV):
            for j in range(nV):
                distance[i][j] = min(distance[i][j], distance[i][k] + distance[k][j])
    return distance

In [ ]:
# update the distance
distance = floyd_warshall(distance)

In [ ]:
# display the updated distance
distance

[[0, 8, 15, 10, 21, 26],
 [8, 0, 12, 7, 16, 23],
 [15, 12, 0, 19, 9, 11],
 [10, 7, 19, 0, 11, 17],
 [21, 16, 9, 11, 0, 13],
 [26, 23, 11, 17, 13, 0]]

In [ ]:
# create a dictionary 'bin_instance' by considering the distance coverage and the updated distance
def binary_distance_(dc,distance_):
  a = {}

  # create the necessary for loops here
  for i in range(len(distance_)):
    for j in range(len(distance_)):
      if distance_[i][j] <= dc:
        a[i,j] = 1
      else:
        a[i,j] = 0

  return a

In [ ]:
a = binary_distance_(11, distance)

In [ ]:
# install PuLP
%pip install pulp==3.3.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 13.8 MB/s  0:00:01
  Attempting uninstall: pulp
    Found existing installation: PuLP 4.2.0
    Uninstalling PuLP-4.2.0:
      Successfully uninstalled PuLP-4.2.0


In [ ]:
#import the PuLP library
from pulp import *

In [ ]:
# create a function to solve the set covering problem
# the parameters required :  bin_distance_ and distance_
def set_covering (a_, distance_):
  # set an empty model
  prob = LpProblem("set_covering", LpMinimize)

  # determine the number of nodes: use the length of distance_
  nb_nodes = len(distance_)

  # declare the decision variables
  x = {}

  for j in range(nb_nodes):
    x[j] = LpVariable(f'x_{j}', cat = 'Binary')

  # set the objective function
  prob += lpSum(x[j] for j in range(nb_nodes))

  # set the constraints
  for i in range(nb_nodes):
    prob += lpSum(a_[i,j]*x[j] for j in range(nb_nodes)) >= 1

  # solve the problem
  prob.solve()

  # get the solutions
  ## get the sol_x here
  sol_x = {}

  for j in range(nb_nodes):
    sol_x[j] = x[j].varValue

  ## get the objective value here. Called it objVal
  objVal = pulp.value(prob.objective)

  return sol_x, objVal

In [ ]:
import time

In [ ]:
# use the set_covering function and also get the CPU time
start_cpu = time.process_time()
solx, objval = set_covering(a, distance)
end_cpu = time.process_time()
cpu_time = end_cpu - start_cpu

In [ ]:
objval

2.0

In [ ]:
solx

{0: 1.0, 1: 0.0, 2: 1.0, 3: 0.0, 4: 0.0, 5: 0.0}

In [ ]:
cpu_time

0.0034937539999972955

# Max Covering Problem

In [ ]:
# the demand list
demand = [10, 8, 22, 18, 7, 55]

In [ ]:
# create a function to solve the max covering problem
## the parameters are bin, distance_ and demand_
def max_covering (a_, distance_, demand_, p_):
  # create an empty model
  prob = LpProblem('max_covering', LpMaximize)

  # get the number of nodes. Use the length of distance_
  nb_nodes = len(distance_)

  # declare the decision variables
  x = {}

  for j in range(nb_nodes):
    x[j] = LpVariable(f'x_{j}', cat = 'Binary')

  z = {}

  for i in range(nb_nodes):
    z[i] = LpVariable(f'z_{i}', cat = 'Binary')

  # set the objective function
  prob += lpSum(demand_[i]*z[i] for i in range(nb_nodes))

  # set the constraints 1
  for i in range(nb_nodes):
    prob += lpSum(a_[i,j]*x[j] for j in range(nb_nodes)) >= z[i]

  # set constraint 2
  prob += lpSum(x[j] for j in range(nb_nodes)) <= p_

  # solve the problem
  prob.solve()

  # get the solution
  sol_z = {}
  sol_x = {}

  ## create the necessary for loops.
  for j in range(nb_nodes):
    sol_x[j] = x[j].varValue

  for i in range(nb_nodes):
    sol_z[i] = z[i].varValue

  ## get the objective value. Call it objVal
  objVal = pulp.value(prob.objective)

  # return
  return sol_z, sol_x, objVal

In [ ]:
import time

In [ ]:
# use the function max_covering and also get the CPU time
start_time = time.process_time()

# coverage distance = 10
a = binary_distance_(11, distance)

# solve the max covering problem
solz, solx, objval = max_covering(a, distance, demand, 2)

# get the end time
end_time = time.process_time()

# capture the cpu time
cpu_time = end_time - start_time

In [ ]:
objval

120.0

In [ ]:
solz

{0: 1.0, 1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0}

In [ ]:
solx

{0: 1.0, 1: 0.0, 2: 1.0, 3: 0.0, 4: 0.0, 5: 0.0}

In [ ]:
# get the cpu time
cpu_time

0.003232702000001808

In [ ]:
# the implementation of the relaxed problem
def relaxed_max_cov (a_, distance_, demand_, p_, lambda_):
  # create an empty model
  prob = LpProblem("relaxed_max_covering", LpMaximize)

  # number of nodes
  nb_nodes = len(demand_)

  # declare the decision variables
  x = {}

  for j in range(nb_nodes):
    x[j] = LpVariable(f'x_{j}', cat = 'Binary')

  z = {}

  for i in range(nb_nodes):
    z[i] = LpVariable(f'z_{i}', cat = 'Binary')

  # set the objective function
  prob += lpSum(demand_[i]*z[i] for i in range(nb_nodes)) + (lpSum(lpSum(a_[i,j]*x[j] - z[i] for j in range(nb_nodes))*lambda_[i] for i in range(nb_nodes)))

  # set the contraint
  prob += lpSum(x[j] for j in range(nb_nodes)) <= p_

  # solve the problem
  prob.solve()

  # get the solution
  sol_z = {}
  sol_x = {}

  ## create the necessary for loops.
  for j in range(nb_nodes):
    sol_x[j] = x[j].varValue

  for i in range(nb_nodes):
    sol_z[i] = z[i].varValue

  ## get the objective value. Call it objVal
  objVal = pulp.value(prob.objective)

  # return
  return sol_z, sol_x, objVal

In [ ]:
p = 2

In [ ]:
# determine the values of lambdas
h_bar = sum(demand)/len(demand)

lambda_ = []

for i in range(len(demand)):
  lambda_.append(h_bar + (demand[i]-h_bar)/2)

In [ ]:
lambda_

[15.0, 14.0, 21.0, 19.0, 13.5, 37.5]

In [ ]:
startTime = time.process_time()

# coverage distance = 11
a = binary_distance_(11, distance)

# use the relaxed max covering problem
solZ, solX, obj_val = relaxed_max_cov(a, distance, demand, p, lambda_)

endTime = time.process_time()

cpuTime = endTime - startTime

In [ ]:
obj_val

133.5

In [ ]:
solZ

{0: 0.0, 1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 5: 0.0}

In [ ]:
solX

{0: 0.0, 1: 0.0, 2: 1.0, 3: 1.0, 4: 0.0, 5: 0.0}

In [ ]:
# implement the fixed covering problem --> lower bound
def fixed_max_cov (a_, distance_, demand_,  p_, solx_):
  # create an empty model
  prob = LpProblem("fixed_max_covering", LpMaximize)

  # define the number nodes
  nb_nodes = len(demand_)

  # declare the decision variables
  z = {}

  for i in range(nb_nodes):
    z[i] = LpVariable(f'z_{i}', cat = 'Binary')

  # set the objective function
  prob += lpSum(demand_[i]*z[i] for i in range(nb_nodes))

  # set the constraints
  for i in range(nb_nodes):
    prob += z[i] <= lpSum(a_[i,j]*solx_[j] for j in range(nb_nodes))

  # solve the problem
  prob.solve()

  # get the solution: solz
  sol_z = {}

  for i in range(nb_nodes):
    sol_z[i] = z[i].varValue

  # get the objective value
  obj_val = pulp.value(prob.objective)

  # return solz, obj_val
  return sol_z, obj_val

In [ ]:
p

2

In [ ]:
sol_z, obj_val = fixed_max_cov(a, distance, demand, p, solX)

In [ ]:
sol_z

{0: 1.0, 1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0}

In [ ]:
obj_val

120.0

In [ ]:
lambda_

[15.0, 14.0, 21.0, 19.0, 13.5, 37.5]

In [ ]:
for i in range(len(lambda_)):
  lambda_[i] = lambda_[i] - 2*(sum(a[i,j]*solX[j] for j in range(len(lambda_))) - sol_z[i])

In [ ]:
lambda_

[15.0, 14.0, 21.0, 19.0, 11.5, 37.5]

In [ ]:
solz, solx, objval = relaxed_max_cov(a, distance,demand,p, lambda_)

In [ ]:
objval

129.5

In [ ]:
solx

{0: 0.0, 1: 0.0, 2: 1.0, 3: 1.0, 4: 0.0, 5: 0.0}

In [ ]:
# get the lower bound
solz, obj_val = fixed_max_cov(a, distance, demand, p, solx)

In [ ]:
obj_val

120.0

In [ ]:
# update the lambdas
for i in range(len(lambda_)):
  lambda_[i] = lambda_[i] - 2*(sum(a[i,j]*solX[j] for j in range(len(lambda_))) - sol_z[i])

In [ ]:
lambda_

[15.0, 14.0, 21.0, 19.0, 9.5, 37.5]

In [ ]:
# get rthe upper bound
solz, solx, objval = relaxed_max_cov(a, distance,demand,p, lambda_)

In [ ]:
objval

126.5

In [ ]:
#  get the lower bound
solz, obj_val = fixed_max_cov(a, distance, demand, p, solx)

In [ ]:
obj_val

84.0

In [ ]:
# implementation of the lagrangian relaxation

## initiate the lower and upper bounds
upperbound = float('inf')
lowerbound = float('-inf')

## initiate the lambdas
h_bar = sum(demand)/len(demand)

lambda_ = []

for i in range(len(demand)):
  lambda_.append(h_bar + (demand[i]-h_bar)/2)

while upperbound > lowerbound:
  # print upper and lowerbounds
  print("Upper Bound ", upperbound)

  print("Lower Bound", lowerbound)

  # get the upperbound using the lambdas
  solz, solx, objval = relaxed_max_cov(a, distance,demand,p, lambda_)

  if upperbound > objval:
    upperbound = objval

  # get the lowerbound using solx
  solz, obj_val = fixed_max_cov(a, distance, demand, p, solx)

  if lowerbound < obj_val:
    lowerbound = obj_val

  # update the lambdas
  for i in range(len(lambda_)):
    lambda_[i] = lambda_[i] - 2*(sum(a[i,j]*solX[j] for j in range(len(lambda_))) - sol_z[i])

Upper Bound  inf
Lower Bound -inf
Upper Bound  133.5
Lower Bound 120.0
Upper Bound  129.5
Lower Bound 120.0
Upper Bound  126.5
Lower Bound 120.0
Upper Bound  124.5
Lower Bound 120.0
Upper Bound  122.5
Lower Bound 120.0
Upper Bound  120.5
Lower Bound 120.0
